# Run Ollama in Colab
---

[![5aharsh/collama](https://raw.githubusercontent.com/5aharsh/collama/main/assets/banner.png)](https://github.com/5aharsh/collama)

This is an example notebook which demonstrates how to run Ollama inside a Colab instance. With this you can run pretty much any small to medium sized models offerred by Ollama for free.

For the list of available models check [models being offerred by Ollama](https://ollama.com/library).


## Before you proceed
---

Since by default the runtime type of Colab instance is CPU based, in order to use LLM models make sure to change your runtime type to T4 GPU (or better if you're a paid Colab user). This can be done by going to **Runtime > Change runtime type**.

While running your script be mindful of the resources you're using. This can be tracked at **Runtime > View resources**.

## Running the notebook
---

After configuring the runtime just run it with **Runtime > Run all**. And you can start tinkering around. This example uses [Llama 3.2](https://ollama.com/library/llama3.2) to generate a response from a prompted question using [LangChain Ollama Integration](https://python.langchain.com/docs/integrations/chat/ollama/).

## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [17]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://cli.github.com/packages stable InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
56 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [18]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [19]:
!ollama pull llama3.2

## And that's it!
---

With this you should be able to freely play around with the models in your scripts. Following is an example using `langchain-ollama` to answer a simple prompt.

If you have a use-case that can help out others feel free to add your notebook to [Collama](https://github.com/5aharsh/collama/fork)

In [20]:
!pip install langchain-ollama

In [27]:
from IPython.display import Markdown

# This cell uses the 'llm' instance created in the 'Initialize and Run' section
question = "What's the length of hypotenuse in a right angled triangle?"
prompt = f"Q: {question} A: Let's think step by step."

if 'llm' in globals() and llm is not None:
    print("Running inference on GPU...")
    response = llm(prompt, max_tokens=256, stop=["Q:"])
    display(Markdown(response['choices'][0]['text']))
else:
    print("❌ Error: 'llm' is not initialized. Please run cell f2d2a311 or a33b9979 first to load the model.")

Llama.generate: 3 prefix-match hit, remaining 23 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture


Running inference on GPU...


llama_perf_context_print:        load time =     447.77 ms
llama_perf_context_print: prompt eval time =      79.25 ms /    23 tokens (    3.45 ms per token,   290.23 tokens per second)
llama_perf_context_print:        eval time =    6133.29 ms /   255 runs   (   24.05 ms per token,    41.58 tokens per second)
llama_perf_context_print:       total time =    6373.53 ms /   278 tokens
llama_perf_context_print:    graphs reused =        253


 The question is asking about the length of hypotenuse. Pythagoras theorem is the formula which is used to find the length of hypotenuse. Pythagoras Theorem states that the square of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the other two sides. Let's denote the length of the hypotenuse as h. So, h2 = a2 + b2. But how to find the values of a and b. In order to find the values of a and b, we need more information about the triangle. So, let's assume that we have a triangle in which the length of one side (let's call it base) is 5 units and the length of the other side (let's call it height) is 12 units. Let's call the hypotenuse as h. In this case, the Pythagoras Theorem would look like this: h2 = a2 + b2. 52 + 122 = h2 25 + 144 = h2 169 = h2 h = √169. h = 13 units. This is the final answer. If you have any question regarding this solution, please let me know. If you

## Optimized Inference with llama-cpp-python
Instead of running the full Ollama service, we can use `llama-cpp-python` compiled with CUDA support for better performance on Colab's T4 GPU.

In [22]:
# Re-attempting the build. This takes a few minutes as it compiles for GPU.
!CMAKE_ARGS="-DGGML_CUDA=ON" pip install llama-cpp-python flask pyngrok

### Download Model
We will now pull the `ALIENTELLIGENCE/footballcoachassistant` model. Since Ollama stores models in a specific blob format, we will use the `ollama` CLI to pull it first, then locate the GGUF file for `llama-cpp-python`.

In [23]:
!ollama pull ALIENTELLIGENCE/footballcoachassistant

### Initialize and Run
Now we load the model using `Llama`. Setting `n_gpu_layers=-1` ensures the entire model is offloaded to the GPU.

In [26]:
from llama_cpp import Llama
import os

# Correct path based on our symlink setup in cell 0d1aa54f
model_path = "/content/models/football_coach.gguf"

llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_gpu_layers=-1
)

output = llm(
    "Q: How should I set up a 4-4-2 formation against a high-pressing team? A:",
    max_tokens=200,
    stop=["Q:", "\n"],
    echo=False
)

print(output["choices"][0]["text"])

llama_model_loader: loaded meta data with 29 key-value pairs and 292 tensors from /content/models/football_coach.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Meta-Llama-3.1
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                            general.license str              = llama3.1
llama_model_loader: - kv   7:         

 “You need to have a good understanding of the opposition, and a good understanding of your own team’s strengths and weaknesses. You need to be able to adjust your tactics accordingly. A high-pressing team is going to be all over you, so you need to be able to deal with that. That means being able to transition quickly from defense to attack, and having a team that can absorb the pressure and then hit back quickly. You need to have a good balance between defending and attacking, and you need to be able to take advantage of any mistakes that the opposition makes. You also need to be able to protect your goalkeeper, so that they can come out for the ball when they need to. If you can manage all of that, then a 4-4-2 formation can be very effective against a high-pressing team. But if you can’t, then you might want to consider a different formation.” 


### 1. Install API Dependencies
To expose the model as an endpoint, we will use `flask` and `pyngrok` (to create a public URL for the local server).

In [ ]:
!pip install flask pyngrok

### 2. Locate and Prepare the Model
Ollama stores models as blobs. We will create a helper script to find the correct GGUF blob for the `footballcoachassistant` model and create a symlink for easier access.

In [ ]:
import os
import json
import time
import subprocess
import threading

# 1. Ensure Ollama Server is running
def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Starting Ollama server...")
threading.Thread(target=run_ollama_serve, daemon=True).start()
time.sleep(5)  # Wait for server to initialize

# 2. Pull the model
print("Pulling model ALIENTELLIGENCE/footballcoachassistant...")
!ollama pull ALIENTELLIGENCE/footballcoachassistant

def get_model_path(model_name):
    base_manifest = "/root/.ollama/models/manifests/registry.ollama.ai"
    parts = model_name.split('/')
    library, model = (parts[0], parts[1]) if len(parts) == 2 else ("library", parts[0])

    possible_paths = [
        os.path.join(base_manifest, library, model, "latest"),
        os.path.join(base_manifest, library, model)
    ]

    for manifest_path in possible_paths:
        if os.path.exists(manifest_path):
            with open(manifest_path, 'r') as f:
                manifest = json.load(f)
                for layer in manifest.get('layers', []):
                    if 'model' in layer.get('mediaType', ''):
                        digest = layer['digest'].replace(':', '-')
                        return f"/root/.ollama/models/blobs/{digest}"
    return None

# 3. Map the blob to GGUF
print("Mapping blob to /content/models/football_coach.gguf...")
time.sleep(2)
model_blob = get_model_path('ALIENTELLIGENCE/footballcoachassistant')

if model_blob and os.path.exists(model_blob):
    !mkdir -p /content/models
    !ln -sf {model_blob} /content/models/football_coach.gguf
    print("✅ Success: Model mapped and ready for Llama-cpp.")
else:
    print("❌ Error: Model mapping failed. Check if the pull command above succeeded.")

### 3. API Endpoint Implementation
This cell initializes the model and starts a Flask server.

**Note:** To access this externally, you would typically need a `pyngrok` auth token, but you can run this locally within the notebook environment for internal requests.

In [ ]:
from llama_cpp import Llama
import os

def initialize_football_coach_model(model_path="/content/models/football_coach.gguf"):
    """
    Loads the football coach assistant model and initializes the Llama instance.
    """
    if not os.path.exists(model_path):
        print(f"❌ Error: Model file not found at {model_path}. Please run the download cell first.")
        return None

    try:
        print(f"Loading model from {model_path}...")
        llm_instance = Llama(
            model_path=model_path,
            n_ctx=2048,           # Context window
            n_gpu_layers=-1,      # Offload everything to GPU
            verbose=False         # Set to True for debugging
        )
        print("✅ Llama instance initialized successfully with GPU acceleration.")
        return llm_instance
    except Exception as e:
        print(f"❌ Failed to initialize Llama instance: {e}")
        return None

# Example usage:
llm = initialize_football_coach_model()

if llm:
    # Simple test run
    resp = llm("Q: Basic strategy for a 4-3-3? A:", max_tokens=50)
    print(f"Test Response: {resp['choices'][0]['text']}")

### 4. Expose the API Externally
1. Get your **Authtoken** from [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken).
2. Replace `'YOUR_NGROK_AUTHTOKEN'` in the cell below.
3. Run the cell to get your public URL.

In [28]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import os

# --- CONFIGURATION ---
print("Please enter your ngrok authtoken (get it from https://dashboard.ngrok.com/):")
NGROK_TOKEN = input("Token: ")
PORT = 5000

app = Flask(__name__)

@app.route('/v1/completions', methods=['POST'])
def api_completions():
    """
    Exposes the globally initialized 'llm' instance via a POST endpoint.
    Expects JSON: {'prompt': '...', 'max_tokens': 150}
    """
    if 'llm' not in globals() or llm is None:
        return jsonify({"error": "Model 'llm' is not initialized on the server."}), 503

    try:
        data = request.json
        if not data or 'prompt' not in data:
            return jsonify({"error": "Missing 'prompt' in request body"}), 400

        prompt = data.get('prompt')
        max_tokens = data.get('max_tokens', 150)
        stop_sequences = data.get('stop', ["Q:", "\n"])

        print(f"Processing request for prompt: {prompt[:50]}...")

        # Generate response using llama-cpp-python instance
        output = llm(
            prompt,
            max_tokens=max_tokens,
            stop=stop_sequences,
            echo=False
        )
        return jsonify(output)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

def start_server():
    app.run(host='0.0.0.0', port=PORT, use_reloader=False)

# --- START NGROK AND FLASK ---
try:
    # Setup ngrok tunnel
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(PORT).public_url
    print(f"\n✅ Public API URL: {public_url}/v1/completions")
    print("Status: Server starting...")

    # Run Flask in a background thread to keep the notebook interactive
    threading.Thread(target=start_server, daemon=True).start()
except Exception as e:
    print(f"❌ Failed to initialize API: {e}")

Please enter your ngrok authtoken (get it from https://dashboard.ngrok.com/):
Token: 341BpO2BlHQbHNzbkxTzXMf66NV_3v4qJ5djFj4cLfYxvD9am

✅ Public API URL: https://lashonda-overfoul-demandingly.ngrok-free.dev/v1/completions
Status: Server starting...


### How to call this API from another service (Example):
```bash
curl -X POST "YOUR_NGROK_URL/v1/completions" \
     -H "Content-Type: application/json" \
     -d '{"prompt": "Q: What is the best defense against a tiki-taka style? A:", "max_tokens": 100}'
```